Installing Phase:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Train:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm
import re

# ==== CONFIG ====
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

IMAGE_SIZE = (100, 300)
TRAIN_IMAGES_PER_FINGER = 3  # Can be changed to 3, 4, etc.

train_images = []
train_labels = []

print("\n🚀 Preparing training data (Strategy 2, no fusion)...")

# Load from both sessions
for session_label, base_path in [("session1", base_path_sess1), ("session2", base_path_sess2)]:
    folder_list = sorted([f for f in os.listdir(base_path) if f.startswith("vein")])

    for folder_name in tqdm(folder_list, desc=f"Processing {session_label}"):
        match = re.match(r"vein(\d{3})_(\d)", folder_name)
        if not match:
            continue

        subject_id = match.group(1)
        finger_id = match.group(2)
        folder_path = os.path.join(base_path, folder_name)

        for img_idx in range(1, TRAIN_IMAGES_PER_FINGER + 1):
            img_path = os.path.join(folder_path, f"{img_idx:02d}.jpg")
            print(f"📸 Loading: {img_path}")  # Added print statement
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                print(f"❌ Failed to load: {img_path}")  # Optional for debugging
                continue

            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            train_images.append(img_norm)
            label = f"{session_label}_subject{subject_id}_fingervein{finger_id}"
            train_labels.append(label)

train_images = np.array(train_images)
train_labels = np.array(train_labels)
print(f"\n✅ Loaded {len(train_images)} individual training samples.")

# ==== (2D)²PCA ====
def compute_2d2pca_projection(images, num_rows, num_cols):
    n = len(images)
    h, w = images[0].shape
    mean_img = sum(images) / n

    G_row = np.zeros((h, h))
    G_col = np.zeros((w, w))

    for img in images:
        A = img - mean_img
        G_row += A @ A.T
        G_col += A.T @ A

    G_row /= n
    G_col /= n

    eig_vals_r, eig_vecs_r = np.linalg.eigh(G_row)
    eig_vals_c, eig_vecs_c = np.linalg.eigh(G_col)

    idx_r = np.argsort(-eig_vals_r)
    idx_c = np.argsort(-eig_vals_c)
    U = eig_vecs_r[:, idx_r[:num_rows]]
    V = eig_vecs_c[:, idx_c[:num_cols]]
    return U, V

# Compute projections
num_row_components = 137
num_col_components = 137
U, V = compute_2d2pca_projection(train_images, num_row_components, num_col_components)

# Project each image
projected_features = []
for img in train_images:
    feat = U.T @ img @ V
    projected_features.append(feat)

# Flatten for classifier
flat_features = np.array([f.flatten() for f in projected_features])
print(f"\n✅ Final PCA feature matrix shape: {flat_features.shape}")


Testing:

In [ ]:
# ==== CONFIG ====
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

# ✅ Define Configurations
NUM_SUBJECTS = 123
NUM_FINGERS = 4
IMAGE_SIZE = (100, 300)

# ✅ Initialize lists
test_data = []
test_labels = []
test_paths = []

for session_label, base_path in [(1, base_path_sess1), (2, base_path_sess2)]:
    for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc=f"Preparing test data - Session {session_label}"):
        for finger_id in range(1, NUM_FINGERS + 1):
            folder_name = f"vein{subject_id:03d}_{finger_id}"
            folder_path = os.path.join(base_path, folder_name)

            for img_idx in [4, 5, 6]:
                img_path = os.path.join(folder_path, f"{img_idx:02d}.jpg")
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

                if img is None:
                    print(f"⚠️ Missing: {img_path}")
                    continue

                print(f"✅ Using: {img_path}")
                img = cv2.resize(img, IMAGE_SIZE)
                img_eq = exposure.equalize_hist(img).astype(np.float64)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

                test_data.append(img_norm)
                label = f"session{session_label}_subject{subject_id:03d}_fingervein{finger_id}"
                test_labels.append(label)
                test_paths.append(img_path)

# Convert to arrays
test_data = np.array(test_data)
test_labels = np.array(test_labels)

# ==== STEP X: PROJECT TEST IMAGES USING (2D)²PCA ====
proj_test_features = [U.T @ img @ V for img in test_data]
flat_test_features = np.array([f.flatten() for f in proj_test_features])

print(f"\n✅ Projected test features shape: {flat_test_features.shape}")
print(f"✅ Total test samples: {len(test_labels)}")


Benchmarking:

In [ ]:
correct_matches = 0
total_tests = len(flat_test_features)

print("\n🔧 Step 1: Classifying test data using Manhattan distance...\n")

for i in range(total_tests):
    test_vec = flat_test_features[i]
    true_label = test_labels[i]

    # Compute Manhattan distances to all train vectors
    distances = np.sum(np.abs(flat_features - test_vec), axis=1)

    # Find nearest neighbor
    min_index = np.argmin(distances)
    predicted_label = train_labels[min_index]

    # Check for exact match
    if predicted_label == true_label:
        correct_matches += 1
        result = "✅ CORRECT"
        emoji = "🎯"
    else:
        result = "❌ WRONG"
        emoji = "⚠️"

    # Output result
    print(f"{emoji} Test sample {i+1}/{total_tests}")
    print(f"    🧾 Predicted: {predicted_label}")
    print(f"    🎯 Actual   : {true_label}")
    print(f"    ➡️  Result   : {result}\n")

# Final accuracy
accuracy = (correct_matches / total_tests) * 100
print("📊 Final Results")
print(f"✅ Correct matches: {correct_matches} / {total_tests}")
print(f"🎯 Recognition Accuracy: {accuracy:.2f}%")